# Distill a small background-removal model from `model_512.onnx`

**Master spec:** `optimize_model_plan/audit.md` (read it first)
**Repo:** https://github.com/octoborg-x/remove-photo-bg

This notebook is the **hand-run Colab** path from spec §3.4. It is resumable:
every cell is idempotent and, where possible, checkpointed. Run top-to-bottom.
If you stop or disconnect, re-run from the top; nothing already done will be
redone.

**Roles:**
- **Teacher** = production `model_512.onnx` (BiRefNet_lite 512 fp32, sigmoid
  baked in). Its outputs are soft probability masks, *already sigmoid* - never
  apply a second sigmoid.
- **Student** = the small ONNX model being distilled (target <= 25 MB, stretch
  <= 10 MB) that must match the teacher's input/output contract:
  `[1,3,S,S]` float32 NCHW + ImageNet mean/std -> `[1,1,S,S]` probabilities.

> NOTE: Colab is ephemeral and GPU quotas apply. Mount Drive (authorised by the
> spec) and persist only manifests, labels, checkpoints, logs, contact sheets
> and the final ONNX to Drive. Do not keep the dataset images on Drive unless
> space remains (spec §4.6).

## 1. Confirm the GPU and runtime

Spec §3.1: run `nvidia-smi` first. Colab's GPU is ephemeral; record it in
`run-metadata.json`. If no GPU is visible, **STOP** - never report CPU numbers
as GPU results (spec §3.1).

In [ ]:
import os, subprocess, sys, json, datetime

print("Hostname:", os.uname().nodename)
print("Python:", sys.version.split()[0])
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvidia-smi | head -12

try:
    import torch
    print("torch:", torch.__version__, "cuda:", torch.version.cuda,
          "cuda_avail:", torch.cuda.is_available())
except Exception as e:
    print("torch:", e)
try:
    import onnxruntime as ort
    print("ort-cpu:", ort.__version__, [
        p for p in ort.get_available_providers()][:6])
except Exception as e:
    print("ort:", e)


In [ ]:
# Pin the exact runtimes for GPU labeling + training, then record in run-metadata.
# Spec §3.7 pins onnxruntime-gpu to 1.31.1 for CUDAExecutionProvider labeling.
# torch is already present in Colab; pin it too so behavior is reproducible.
%pip install -q --upgrade "onnxruntime-gpu==1.31.1" pillow numpy "torch>=2.2" "torchvision>=0.17"
%pip install -q onnx onnxruntime onnxoptimizer onnxsim

import os, subprocess, sys, json, datetime
import onnxruntime as ort, torch

GPU = torch.cuda.is_available()
ORT_GPU = "CUDAExecutionProvider" in ort.get_available_providers()
print("GPU:", GPU, "| ORT CUDA:", ORT_GPU)
if not ORT_GPU:
    raise SystemExit("No CUDAExecutionProvider in onnxruntime-gpu. STOP (§3.1).")

meta = {
    "date": datetime.datetime.utcnow().isoformat() + "Z",
    "hostname": os.uname().nodename,
    "gpu": None, "driver": None, "cuda": None,
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "onnxruntime_gpu": ort.__version__,
}
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version",
                          "--format=csv,noheader,nounits"],
                         capture_output=True, text=True, check=True)
    name, drv = out.stdout.strip().split(",")
    meta["gpu"], meta["driver"] = name.strip(), drv.strip()
except Exception as e:
    print("nvidia-smi parse:", e)
meta["cuda"] = torch.version.cuda if torch.cuda.is_available() else None
os.makedirs("smoke-results/distill", exist_ok=True)
with open("smoke-results/distill/run-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))
print("Saved smoke-results/distill/run-metadata.json")


## 2. Mount Drive, clone the repo, fetch the teachers

Spec §3.5, §3.6: Drive is authorised. Persist **manifests, labels (tar),
checkpoints, logs, contact sheets, final ONNX** to Drive. Clone the repo fresh
each session (or use an existing clone in /content).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
DRIVE = "/content/drive/MyDrive/remove-photo-bg-distill"
os.makedirs(DRIVE, exist_ok=True)

# Space check (spec §4.1)
!df -h /content | tail -1
total, used, free = shutil.disk_usage("/content/drive")
print("Drive free GB:", round(free / (1<<30), 1))


In [ ]:
import os, subprocess

REPO = "/content/remove-photo-bg"
REPO_URL = "https://github.com/octoborg-x/remove-photo-bg.git"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
os.chdir(REPO)
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())


In [ ]:
# Fetch the teacher model(s) by pinned SHA-256 (spec §1, §3).
# Teacher 512 (production, ~185 MB) is REQUIRED. Teacher 1024 (optional,
# labeling-only) is downloaded only if you want the sharper labels.
import hashlib, os, urllib.request

MODELS = {
    "512-fp32": {
        "dest": "public/models/model_512.onnx",
        "sha256": "eaafaa10f19c63b77e2bae77bf4b2f59054c208ea08b8c6648f95ef36a3b6b87",
        "bytes": 193514682,
        "urls": [
            "https://huggingface.co/Phoenix-ActuallyFree/BiRefNet_lite_512-ONNX/resolve/main/onnx/model.onnx",
        ],
    },
    "1024-fp16": {
        "dest": "/tmp/hf-model/model_1024_fp16.onnx",
        "sha256": "d39b897ceb16ae654c1731f3dba0cf9b368d9cae74b5a57459b455cc8bfec402",
        "bytes": 114538221,
        "urls": [
            "https://huggingface.co/onnx-community/BiRefNet_lite-ONNX/resolve/"
            "de15b22ba131738a16dff04aab8bdf8dc32e3ac1/onnx/model_fp16.onnx",
            "https://huggingface.co/onnx-community/BiRefNet_lite-ONNX/resolve/"
            "de15b22ba131738a16dff04aab8bdf8dc32e3ac1/model_fp16.onnx",
        ],
    },
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for label, spec in MODELS.items():
    dest = spec["dest"]
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.isfile(dest) and sha256_file(dest) == spec["sha256"]:
        print(f"{label}: already present, hash ok")
        continue
    ok = False
    for url in spec["urls"]:
        try:
            print(f"{label}: downloading {url}")
            urllib.request.urlretrieve(url, dest)
            if sha256_file(dest) == spec["sha256"]:
                ok = True
                break
            else:
                print(f"{label}: hash mismatch, trying next url")
                os.remove(dest)
        except Exception as e:
            print(f"{label}: {url} failed: {e}")
    if not ok:
        print(f"FAILED to fetch {label}; run npm run fetch-models locally and copy the file.")
        if label == "512-fp32":
            raise SystemExit("Teacher 512 is required. STOP.")
print("models ready")
!du -sh public/models/* 2>/dev/null


## 3. Teacher smoke + GPU<->CPU parity

Before generating labels, verify the teacher runs and that **GPU and CPU masks
agree** (spec §3.7: IoU >= 0.995 and mean abs diff <= 0.005 on the 3 test
images). This catches driver/onnxruntime mismatches early. CPU-vs-CPU parity is
>= 0.999; fp32 GPU differs by rounding.

In [ ]:
import numpy as np, onnxruntime as ort, os
from PIL import Image

def app_preprocess(img, size=512):
    """App-exact preprocessing: resize, /255, ImageNet normalize. Spec §2."""
    img = img.convert("RGB").resize((size, size), Image.BILINEAR)
    x = np.asarray(img, np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], np.float32)
    std = np.array([0.229, 0.224, 0.225], np.float32)
    x = (x - mean) / std
    return x.transpose(2, 0, 1)[None]                    # [1,3,S,S] NCHW

def run_mask(model_path, img, size=512, providers=None):
    sess = ort.InferenceSession(model_path, providers=providers or ort.get_available_providers())
    iname = sess.get_inputs()[0].name
    outname = sess.get_outputs()[0].name
    x = app_preprocess(img, size)
    y = sess.run([outname], {iname: x})[0]              # [1,1,S,S]
    return y[0, 0]                                       # probability, sigmoid already applied

def iou(a, b, th=0.5):
    a, b = (a > th), (b > th)
    inter = (a & b).sum()
    union = (a | b).sum()
    return (inter / union) if union else 1.0

if os.path.isdir("test-images"):
    TEST = sorted(os.path.join("test-images", f) for f in os.listdir("test-images")
                  if f.lower().endswith((".png", ".jpg", ".jpeg")))[:3]
else:
    TEST = []
print("test images:", TEST)
if TEST:
    cpup, gpup = ["CPUExecutionProvider"], ["CUDAExecutionProvider", "CPUExecutionProvider"]
    for p in TEST:
        img = Image.open(p)
        mcpu = run_mask("public/models/model_512.onnx", img, providers=cpup)
        mgpu = run_mask("public/models/model_512.onnx", img, providers=gpup)
        d = np.abs(mcpu - mgpu)
        print(f"{os.path.basename(p)}: IoU={iou(mcpu, mgpu):.5f} meanAbs={d.mean():.6f} maxAbs={d.max():.6f}")
    print("GPU parity check done.")
else:
    print("No test-images dir in repo; upload the 3 test images to /content and rerun.")


## 4. Data manifest + teacher soft labels

**Phase 1 (spec §6-Phase1) - STOP FOR REVIEW.** The spec requires a written
data plan (`DISTILL_PLAN.md`) and **your approval before training**. This cell
writes `data/manifest.csv` with per-image licence filtering (sources per spec
§5). You must review the manifest and the spec's Phase-1 STOP gate before
running Phase 2.

> The master spec requires **10,000 to 30,000** training images, licence-
> filtered. This is the part you must choose/approve. This notebook provides a
> manifest skeleton for the approved sources (Open Images V7 CC-BY, Smithsonian
> CC0, Met PD, Wikimedia CC0/PD) with a strict licence gate. **Only run after
> reading spec §5 and §6-Phase1** and confirming sources.

In [ ]:
# Manifest columns required by spec §5:
header = ["id","source","url","licence","licence_url","author",
          "attribution_text","date_checked","sha256","width","height",
          "category","tier","split","reject_reason"]
os.makedirs("data", exist_ok=True)
import csv
if not os.path.isfile("data/manifest.csv"):
    with open("data/manifest.csv", "w", newline="") as f:
        csv.writer(f).writerow(header)
    print("Wrote data/manifest.csv (empty skeleton).")
else:
    print("data/manifest.csv exists; won't overwrite.")
print("NEXT: implement scripts/distill/fetchers/ for the sources you approve, then STOP for review")


In [ ]:
# Teacher soft labels (spec §6-Phase2) - app-exact preprocessing, no threshold.
# SOFT probability masks saved as 8-bit PNG at 512. Resumable: skip already-done.
import os, numpy as np, onnxruntime as ort
from PIL import Image
from pathlib import Path

LABEL_ROOT = os.path.join(DRIVE, "labels512")
os.makedirs(LABEL_ROOT, exist_ok=True)

labels_done = set(p.stem for p in Path(LABEL_ROOT).glob("*.png"))
print(f"Labels already present: {len(labels_done)}")

# For every row in data/manifest.csv that is tier A/B and split=train/val,
# fetch the image, run teacher, save soft 8-bit mask. Full implementation goes
# in scripts/distill/make-labels.py; this is the inline runner.
import csv
rows = []
if os.path.isfile("data/manifest.csv"):
    with open("data/manifest.csv") as f:
        rows = list(csv.DictReader(f))
    print("manifest rows:", len(rows))

# --- placeholder pipeline; replace with real per-source download ---
# For each approved row: download to /content, teacher-forward, save
# {id}_mask.png (8-bit, 0-255 = probability*255). Verify 8-bit quantisation
# changes IoU < 0.001 on 200 images first (spec §4.3).
print("Teacher-label generation skeleton - wire to scripts/distill/make-labels.py")


## 5. Composite generation (Tier-A only)

Used *on the fly* by the dataloader (spec §6-Phase2 `composite.py`): estimate
clean foreground colour, paste onto approved CC0 backgrounds, add shadows/noise,
then **round-trip filter** (teacher on the composite must agree with the known
alpha; discard if IoU < 0.97). Composites only supplement real images; never
evaluate on composites (spec §6-Phase2).

In [ ]:
# Skeleton for scripts/distill/composite.py; on-the-fly in the dataloader.
# This cell just verifies pymatting installs (needed for foreground colour estimate).
%pip install -q pymatting
import pymatting
print("pymatting imported ok")


## 6. Students

**Student A** (baseline, no training): U2-Netp from its official source -
measure as the quality floor (spec §6-Phase3).

**Student B** (primary candidate): UNet/FPN decoder on a mobile encoder
(MobileNetV3-Large or EfficientNet-Lite) with high-res skip connections. All ops
must export cleanly to ONNX opset 17 and run on onnxruntime-web WASM: no
deformable conv, no large attention, no exotic ops. BatchNorm folded at export
(spec §6-Phase3).

In [ ]:
# PINNED dependencies for the train pipeline. Pin in requirements-distill.txt too.
%pip install -q torch torchvision timm albumentations pyyaml
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__)


In [ ]:
# Student B - define the model here. This is the architecture to train.
# The exact channels/decoder are a placeholder you tune; the contract must hold:
#   input [1,3,S,S] float32 NCHW -> output [1,1,S,S] float32 sigmoid already baked.
import torch, torch.nn as nn

def build_student(backbone="mobilenetv3_large_100", in_ch=3, out_ch=1):
    import timm
    enc = timm.create_model(backbone, features_only=True, out_indices=(2,3,4),
                            pretrained=True)
    # MobileNetV3 features_only returns C = (24, 48, 96) or similar.
    # Choose a simple decoder: upsample + conv per scale, concatenate, final conv
    # + sigmoid. This is a placeholder; replace with a real UNet/FPN decoder that
    # preserves edges. Keep ops ONNX-opset17-WASM-friendly (no deform conv, no
    # large attention).
    raise NotImplementedError(
        "Implement the UNet/FPN decoder. The exact architecture is your choice.")

print("Student B building block wired.")


## 7. Train the student (CPU smoke -> GPU real)

Spec §6-Phase3: do a **CPU smoke run** first (epoch on a few images, loss must
decrease), then run the real training on the Colab GPU. Loss =
`BCE(student, teacher_soft) + IoU loss + 0.5*L1(grad) + 0.5*SSIM` (+ optional
boundary-weight and GT-BCE where GT exists). AdamW + cosine, mixed precision,
checkpoint every epoch **and every 10 minutes**, resumable, seeded.
Checkpoints go to Drive (keep last 3 + best).

In [ ]:
# CPU smoke: overfit on a tiny subset. Must show loss decreasing.
# If this cell does not complete in minutes with a decreasing loss, fix the model
# before spending GPU time. (Implementation uses scripts/distill/train.py with a
# YAML config; this is the inline runner.)
print("SMOKE training placeholder. Wire scripts/distill/train.py here.")
print("Requirements: loss decreases, resumable, seeded, checkpoint every 10 min.")


In [ ]:
# Real training on GPU.
#   - Load student, teacher labels from Drive (labels512/),
#   - Mixed precision (torch.autocast), AdamW, cosine schedule,
#   - checkpoint every epoch AND every 10 min, Drive path =
#       DRIVE/checkpoints/{step}.pt keep last 3 + best,
#   - log to DRIVE/logs/train.log,
#   - STOP and report after a first short GPU run; continue only if you approve
#     (spec §6-Phase3 STOP).
print("Real GPU training placeholder - see scripts/distill/train.py + config.yaml")


## 8. Export to ONNX and optimize

Export the trained student at fixed S, opset 17, **sigmoid included**; check
names/shapes/dtype/finiteness. Then, per spec §6-Phase4, optimization
experiments each reported **separately**:
- shape inference + onnxsim (pinned): accept only if max abs output error <= 1e-5
- weight-only fp16 (fp16 initializers + Cast folded at load; fp32 compute) -
  **never end-to-end fp16 on WASM** (measured ~51x slower)
- brotli/gzip compression measured separately
- int8: skip unless time allows; document rejection otherwise
(Spec §6-Phase4; also the engineer's audit in optimize-model_512/).

In [ ]:
# Export + optimize. Placeholder - full logic in scripts/distill/export.py.
import onnx
print("Export ONNX skeleton. Must include sigmoid; check I/O contract, "
      "names, shapes, dtype, finiteness.")


## 9. Evaluate every candidate against the teacher

Spec §6-Phase4: for each candidate, same production pipeline+threshold:
IoU@0.5, Dice, MAE, MSE, boundary F-score, mean abs alpha error in the 0.05-0.95
band, foreground precision/recall, false-foreground/background pixels, per image,
per category, aggregate. Then a **separate** diagnostic threshold sweep
(0.30 ... 0.70). Never tune threshold per candidate to hide regressions.
Save per eval image: original, teacher mask, student mask, absolute-diff map,
checkerboard cutout, zoomed edge crops; write a per-image CSV.

In [ ]:
# Evaluation placeholder - scripts/distill/evaluate.py.
# Summary of what this must do:
#   - load eval set (real images only; no composites),
#   - for each candidate vs teacher: compute IoU/Dice/MAE/MSE/boundary-F/alpha-band,
#   - write smoke-results/distill/metrics.csv + visual contact sheets,
#   - print per-category + aggregate.
print("Evaluation placeholder.")


## 10. Acceptance (spec §7) and report

A candidate may replace production only if **ALL** hold:
- Mean IoU >= 0.97 vs teacher; no eval image below 0.90.
- No visible failure on glass/hair/fur/thin structures; boundary F within an
  agreed margin of teacher in those categories.
- Valid transparent PNG at original dimensions through the existing pipeline.
- File size <= 25 MB (fp32 or weight-only fp16); >= 70% smaller than 185 MB.
- WASM warm latency <= teacher's (phone ref ~6.6 s), target <= 2 s. WebGPU never
  default without per-hardware correctness validation.
- No new crashes/memory failures; existing tests + smoke tests pass.
- `DISTILL_PROVENANCE.md` complete, every licence verified.
If not met, **NO-GO**, FP32 stays in production.

Deliverables (spec §8): `DISTILL_PROVENANCE.md`, `DISTILL_PLAN.md`,
`DISTILL_RESULTS.md`, `scripts/distill/`, `requirements-distill.txt`, Colab
notebooks, `run-metadata.json`, tests, per-image CSV, visual comparisons,
verdict GO / GO-WITH-FALLBACK / NO-GO.

In [ ]:
# Print the current run metadata + drive usage for the report.
import json, os, shutil
meta = json.load(open("smoke-results/distill/run-metadata.json"))
print(json.dumps(meta, indent=2))
used = shutil.disk_usage("/content/drive").used // (1 << 30)
print("Drive used GB:", used)
lp = os.path.join(DRIVE, "labels512")
print("Labels on Drive:", len(os.listdir(lp)) if os.path.isdir(lp) else 0)
print("Notebook complete. See spec §6-Phase4 and §8 for the full report.")
